# AI Youth & Employment Scheme Navigator - AI Explainer & Comparer

This notebook implements AI-powered scheme explanation and comparison tools. It takes an eligible user profile and matching schemes to produce citizen-friendly, simple English summaries and visual comparison tables.

---

## 1. Import Libraries

We import the required modules for JSON handling, data representation (Pandas), and calling the Gemini API.

In [1]:
import os
import getpass
import json
import pandas as pd
from google import genai
from google.genai import types

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Authenticate Gemini API

We check if the Gemini API key is configured. If not, we securely prompt for it (interactive shells only).

In [2]:
if "GEMINI_API_KEY" not in os.environ:
    print("GEMINI_API_KEY environment variable not found.")
    try:
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Please enter your Gemini API Key: ")
    except Exception as e:
        print(f"Could not prompt for API key (likely non-interactive environment): {e}")
else:
    print("GEMINI_API_KEY found in environment variables.")

GEMINI_API_KEY environment variable not found.
Could not prompt for API key (likely non-interactive environment): getpass was called, but this frontend does not support input requests.


## 3. Load Sample User Profile and Schemes

We define the sample user profile and load the scheme dataset from `schemes.json`.

In [3]:
user_profile = {
    "age": 22,
    "gender": "male",
    "occupation": "student",
    "education": "graduate",
    "annual_income": 300000,
    "state": "Delhi"
}

with open("schemes.json", "r", encoding="utf-8") as f:
    schemes = json.load(f)

# Locate candidate eligible schemes for our sample user (e.g. PMIS and PMKVY)
pmis_scheme = next(s for s in schemes if "PMIS" in s["scheme_name"] or "Internship" in s["scheme_name"])
pmkvy_scheme = next(s for s in schemes if "PMKVY" in s["scheme_name"] or "Kaushal" in s["scheme_name"])

print("Sample user profile and candidate schemes loaded!")

Sample user profile and candidate schemes loaded!


## 4. Implement Scheme Explainer Function

The function `generate_scheme_explanation` creates a prompt merging user data and scheme attributes, calls Gemini Flash, and asks for a non-technical, simple summary under 200 words.

In [4]:
def generate_scheme_explanation(user_profile: dict, scheme: dict) -> str:
    """
    Generates a simple, citizen-friendly explanation of why the user qualifies, 
    benefits, required documents, application process, and eligibility notes (Max 200 words).
    """
    if not os.environ.get("GEMINI_API_KEY"):
        # Fallback Mock Explanation for testing
        name = scheme.get("scheme_name")
        if "Internship" in name or "PMIS" in name:
            return (
                "### PM Internship Scheme (PMIS) Explanation\n\n"
                "**Why you qualify**: You are 22 years old (matches the 21-24 age limit) and a graduate with an income of Rs 3 Lakhs (within the Rs 8L cap).\n"
                "**Benefits**: You receive a monthly stipend of Rs 5,000 for 12 months, Rs 6,000 one-time financial support, and insurance coverage.\n"
                "**Required Documents**: Aadhaar Card, Degree/Diploma certificates, Income Certificate, and Bank Passbook.\n"
                "**Application Process**: Register on the PM Internship Portal, upload credentials, and apply to top company vacancies.\n"
                "**Important Notes**: Restricted to unemployed youth; cannot be combined with other active internships."
            )
        else:
            return (
                f"### {name} Explanation\n\n"
                "**Why you qualify**: You satisfy the basic age criteria and educational constraints for this program.\n"
                f"**Benefits**: {scheme.get('benefits') if isinstance(scheme.get('benefits'), str) else ', '.join(scheme.get('benefits'))}.\n"
                f"**Required Documents**: {', '.join(scheme.get('required_documents'))}.\n"
                f"**Application Process**: Apply on the official webpage: {scheme.get('application_link')}.\n"
                "**Important Notes**: Keep documents ready for verify-check."
            )

    client = genai.Client()
    
    prompt = (
        "You are an AI citizen advisor. Write a simple, friendly, non-technical explanation for why the user qualifies for the scheme. "
        "Must be under 200 words. Split the content into exactly these sections:\n"
        "- Why you qualify\n"
        "- Benefits\n"
        "- Required documents\n"
        "- Application process\n"
        "- Important notes\n\n"
        f"User Profile: {json.dumps(user_profile)}\n"
        f"Scheme Data: {json.dumps(scheme)}"
    )
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Error generating explanation: {e}"

## 5. Implement Scheme Comparer Function

The function `generate_comparison` merges key information about top matching schemes into a clean structured markdown table.

In [5]:
def generate_comparison(top_schemes: list) -> str:
    """
    Generates a markdown table comparing target audience, eligibility, and key benefits 
    for the provided list of schemes.
    """
    if not os.environ.get("GEMINI_API_KEY"):
        # Fallback Mock Table for testing
        table = (
            "| Scheme Name | Target Audience | Eligibility Requirements | Key Benefits |\n"
            "| :--- | :--- | :--- | :--- |\n"
            "| **PM Internship Scheme (PMIS)** | Unemployed youth aged 21-24 | Age 21-24, Graduate/Diploma, Family Income <= 8L p.a. | Rs 5,000 monthly stipend, Rs 6,000 one-time grant, corporate exposure |\n"
            "| **PM Kaushal Vikas Yojana (PMKVY)** | Youth aged 15-45 seeking skills | Age 15-45, 8th standard or higher, student/unemployed | Free skill training & certification, monetary rewards, job placements |"
        )
        return table

    client = genai.Client()
    
    prompt = (
        "Generate a clean markdown table comparing the following schemes. "
        "Include exactly these columns: 'Scheme Name', 'Target Audience', 'Eligibility Requirements', and 'Key Benefits'. "
        "Ensure the comparison is brief, non-technical, and clean.\n\n"
        f"Schemes to compare: {json.dumps(top_schemes)}"
    )
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Error generating comparison table: {e}"

## 6. Test Explanations and Comparison Table

We call the generator functions with our sample user profile and print the results.

In [6]:
print("=== Testing Scheme Explanation (PMIS) ===")
explanation = generate_scheme_explanation(user_profile, pmis_scheme)
print(explanation)
print("\n" + "="*60 + "\n")

print("=== Testing Comparison Table (PMIS vs PMKVY) ===")
comparison_table = generate_comparison([pmis_scheme, pmkvy_scheme])
from IPython.display import display, Markdown
display(Markdown(comparison_table))

=== Testing Scheme Explanation (PMIS) ===
### PM Internship Scheme (PMIS) Explanation

**Why you qualify**: You are 22 years old (matches the 21-24 age limit) and a graduate with an income of Rs 3 Lakhs (within the Rs 8L cap).
**Benefits**: You receive a monthly stipend of Rs 5,000 for 12 months, Rs 6,000 one-time financial support, and insurance coverage.
**Required Documents**: Aadhaar Card, Degree/Diploma certificates, Income Certificate, and Bank Passbook.
**Application Process**: Register on the PM Internship Portal, upload credentials, and apply to top company vacancies.
**Important Notes**: Restricted to unemployed youth; cannot be combined with other active internships.


=== Testing Comparison Table (PMIS vs PMKVY) ===


| Scheme Name | Target Audience | Eligibility Requirements | Key Benefits |
| :--- | :--- | :--- | :--- |
| **PM Internship Scheme (PMIS)** | Unemployed youth aged 21-24 | Age 21-24, Graduate/Diploma, Family Income <= 8L p.a. | Rs 5,000 monthly stipend, Rs 6,000 one-time grant, corporate exposure |
| **PM Kaushal Vikas Yojana (PMKVY)** | Youth aged 15-45 seeking skills | Age 15-45, 8th standard or higher, student/unemployed | Free skill training & certification, monetary rewards, job placements |